# 02 — Baseline & Error-Analysis Figures (Ch 4.1, 4.2)

**Updated 2026-05-31 after WS1 completion:**
- Per-query mDPR data now loaded from canonical CSV (`baseline_dense_per_query_with_length.csv`), not the buggy `exp_001_quantitative_analysis.json`.
- Failure threshold annotation auto-updates: ~34% (vs 39% claimed in old §4.2).
- Length bins: **1–3 / 4–8 / 9+ words** (Scheme A).
- Table 4.1 rounds to 3 decimals (Decision D4): mDPR 0.499, BM25 0.462.

**Outputs:**
- Table 4.1 (LaTeX)
- Fig 4.1 v1/v2 — per-query NDCG@10 distribution (hist/CDF)
- Fig 4.2 v1/v2 — failure cliff (CDF with annotation / bucket bar)
- Fig 4.3 v1/v2/v3 — NDCG@10 by query length (box/violin/scatter)
- Fig 4.4 — Recall@k curve k=1…100

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import *

## Table 4.1 — Baselines (mDPR vs BM25)

In [ ]:
mdpr = load_json(DATA_RAW / 'baseline_dense_metrics.json')
bm25 = load_json(DATA_RAW / 'baseline_bm25_metrics.json')

def pick(d, keys):
    out = {}
    for k in keys:
        out[k] = d.get(k) or d.get(k.replace('@', '_').lower()) or d.get(k.lower())
    return out

rows = [
    {'Retriever': 'mDPR (Dense)',  **pick(mdpr, ['Recall@10', 'Recall@100', 'NDCG@10', 'MRR'])},
    {'Retriever': 'BM25S (Sparse)', **pick(bm25, ['Recall@10', 'Recall@100', 'NDCG@10', 'MRR'])},
]
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

# Decision D4: round to 3 decimals (matches published rounding)
latex = tbl.to_latex(index=False, float_format='%.3f', column_format='lcccc')
(OUTPUT_PDF / 'table_4_1.tex').write_text(latex, encoding='utf-8')
print('saved: table_4_1.tex')

## Per-query data (canonical, post-WS1)

Source: `data/computed/baseline_dense_per_query_with_length.csv` — produced from the canonical TREC run with `pytrec_eval`, replacing the old buggy `exp_001_quantitative_analysis.json`.

In [ ]:
# Canonical per-query mDPR data (computed by notebook 01 from TREC + pytrec_eval).
# The earlier 'baseline_dense_per_query_with_length.csv' was built from a buggy analysis JSON;
# WS1 §2 reconciliation found it disagrees with the canonical run. We use the canonical here.
mdpr_pq = pd.read_csv(DATA_COMPUTED / 'per_query_baseline_dense.csv', dtype={'qid': str})
ql = pd.read_csv(DATA_COMPUTED / 'query_lengths.csv', dtype={'qid': str})
mdpr_pq = mdpr_pq.merge(ql[['qid', 'word_count']], on='qid', how='left')
mdpr_pq['bucket_a'] = mdpr_pq['word_count'].apply(length_bin)
mdpr_pq['bucket_a'] = pd.Categorical(mdpr_pq['bucket_a'], categories=LENGTH_BIN_ORDER, ordered=True)
print(f'mDPR per-query (canonical): {len(mdpr_pq)} queries · mean nDCG@10={mdpr_pq.ndcg10.mean():.4f}')
print(mdpr_pq.groupby('bucket_a', observed=True).agg(n=('ndcg10','size'), mean=('ndcg10','mean')))

# BM25 per-query also from notebook 01 (canonical)
bm25_pq_path = DATA_COMPUTED / 'per_query_baseline_bm25.csv'
bm25_pq = pd.read_csv(bm25_pq_path, dtype={'qid': str}) if bm25_pq_path.exists() else None
if bm25_pq is None:
    print('BM25 per-query missing — run notebook 01 first')
else:
    print(f'BM25 per-query: {len(bm25_pq)} queries · mean nDCG@10={bm25_pq.ndcg10.mean():.4f}')


## Fig 4.1 — NDCG@10 distribution

In [ ]:
# v1 — overlaid histograms
fig, ax = plt.subplots()
bins = np.linspace(0, 1, 26)
ax.hist(mdpr_pq.ndcg10, bins=bins, alpha=0.55, label='mDPR (Dense)', color='#1f1f1f', edgecolor='white')
if bm25_pq is not None:
    ax.hist(bm25_pq.ndcg10, bins=bins, alpha=0.55, label='BM25', color='#8c8c8c', edgecolor='white')
ax.set_xlabel('NDCG@10')
ax.set_ylabel('Number of queries')
ax.legend(loc='upper right')
save_fig(fig, 'fig_4_1_ndcg_hist_v1')

In [ ]:
# v2 — empirical CDF
fig, ax = plt.subplots()
def ecdf(x):
    x = np.sort(x)
    return x, np.arange(1, len(x) + 1) / len(x)
xs, ys = ecdf(mdpr_pq.ndcg10)
ax.step(xs, ys, where='post', label='mDPR (Dense)', color='#1f1f1f')
if bm25_pq is not None:
    xs, ys = ecdf(bm25_pq.ndcg10)
    ax.step(xs, ys, where='post', label='BM25', color='#8c8c8c', linestyle='--')
ax.set_xlabel('NDCG@10')
ax.set_ylabel('Cumulative fraction of queries')
ax.legend(loc='lower right')
save_fig(fig, 'fig_4_1_ndcg_cdf_v2')

## Fig 4.2 — Failure cliff

**Note:** annotation auto-computes the failure rate from canonical data. Post-WS1 value: ~34% (was 39% in the buggy JSON).

In [ ]:
# v1 — CDF with annotation at NDCG@10 = 0.3
fig, ax = plt.subplots()
xs, ys = ecdf(mdpr_pq.ndcg10)
ax.step(xs, ys, where='post', color='#1f1f1f')
thresh = 0.3
failure_rate = (mdpr_pq.ndcg10 < thresh).mean()
ax.axvline(thresh, color='#4d4d4d', linestyle=':', linewidth=1)
ax.axhline(failure_rate, color='#4d4d4d', linestyle=':', linewidth=1)
ax.annotate(f'{failure_rate*100:.1f}% of queries\nNDCG@10 < {thresh}',
            xy=(thresh, failure_rate), xytext=(0.45, 0.25),
            arrowprops=dict(arrowstyle='->', color='#4d4d4d'), fontsize=9)
ax.set_xlabel('NDCG@10')
ax.set_ylabel('Cumulative fraction of queries')
save_fig(fig, 'fig_4_2_failure_cliff_v1')
print(f'Failure rate (canonical): {failure_rate*100:.1f}% — was 39% in the buggy file.')

In [ ]:
# v2 — failure rate per NDCG bucket
fig, ax = plt.subplots()
edges = np.arange(0, 1.05, 0.1)
labels = [f'[{edges[i]:.1f},{edges[i+1]:.1f})' for i in range(len(edges) - 1)]
counts, _ = np.histogram(mdpr_pq.ndcg10, bins=edges)
props = counts / counts.sum()
ax.bar(labels, props, color='#4d4d4d', edgecolor='black')
ax.set_xlabel('NDCG@10 bucket')
ax.set_ylabel('Fraction of queries')
plt.xticks(rotation=45, ha='right')
save_fig(fig, 'fig_4_2_failure_buckets_v2')

## Fig 4.3 — NDCG@10 by query length (Scheme A: 1–3 / 4–8 / 9+ words)

The bucket column in the CSV uses an older `<5 / 5–9 / ≥1`0 scheme; we re-bucket in-notebook using `length_bin()` (Scheme A) so the figure matches §3.3 and §4.10 exactly.

In [ ]:
df = mdpr_pq  # already has bucket_a column from earlier cell

# v1 box
fig, ax = plt.subplots()
df.boxplot(column='ndcg10', by='bucket_a', ax=ax, grid=False, patch_artist=True,
           boxprops=dict(facecolor='#b3b3b3', color='#1f1f1f'),
           medianprops=dict(color='#1f1f1f'), widths=0.5)
ax.set_title(''); fig.suptitle('')
ax.set_xlabel('Query length (words)')
ax.set_ylabel('NDCG@10')
save_fig(fig, 'fig_4_3_length_box_v1')

# v2 violin
fig, ax = plt.subplots()
data = [df.loc[df.bucket_a == b, 'ndcg10'].values for b in LENGTH_BIN_ORDER]
parts = ax.violinplot(data, showmedians=True, widths=0.7)
for pc in parts['bodies']:
    pc.set_facecolor('#b3b3b3'); pc.set_edgecolor('#1f1f1f')
ax.set_xticks(range(1, len(LENGTH_BIN_ORDER) + 1))
ax.set_xticklabels(LENGTH_BIN_ORDER)
ax.set_xlabel('Query length (words)')
ax.set_ylabel('NDCG@10')
save_fig(fig, 'fig_4_3_length_violin_v2')

# v3 scatter with binned means trendline
fig, ax = plt.subplots()
ax.scatter(df.word_count, df.ndcg10, alpha=0.15, s=8, color='#4d4d4d')
means = df.groupby('word_count').ndcg10.mean().reset_index()
means = means[means.word_count <= 20]
ax.plot(means.word_count, means.ndcg10, color='#1f1f1f', linewidth=2, label='Mean NDCG@10 per length')
ax.set_xlabel('Query length (words)')
ax.set_ylabel('NDCG@10')
ax.legend()
ax.set_xlim(0, 20)
save_fig(fig, 'fig_4_3_length_scatter_v3')

## Fig 4.4 — Recall@k curve (k=1…100) from canonical TREC runs

In [ ]:
import pytrec_eval
qrels = load_json(DATA_RAW / 'miracl_qrels_dev.json')
qrels_str = {str(q): {str(d): int(r) for d, r in v.items()} for q, v in qrels.items()}
ks = list(range(1, 101))
measures = {f'recall.{k}' for k in ks}

def avg_recall_curve(run_path):
    df = load_trec_run(run_path)
    run = {}
    for qid, g in df.groupby('qid'):
        run[str(qid)] = {str(d): float(s) for d, s in zip(g['docid'], g['score'])}
    ev = pytrec_eval.RelevanceEvaluator(qrels_str, measures)
    res = ev.evaluate(run)
    return [np.mean([r[f'recall_{k}'] for r in res.values()]) for k in ks]

curves = {}
for name, fname in [('mDPR (Dense)', 'baseline_dense_run.txt'),
                     ('BM25', 'baseline_bm25_run.txt')]:
    path = DATA_RAW / fname
    if path.exists():
        print(f'computing recall curve for {name}...')
        curves[name] = avg_recall_curve(path)
    else:
        print(f'  SKIP {name}: {fname} missing')

if curves:
    fig, ax = plt.subplots()
    styles = {'mDPR (Dense)': dict(color='#1f1f1f'), 'BM25': dict(color='#8c8c8c', linestyle='--')}
    for name, ys in curves.items():
        ax.plot(ks, ys, label=name, **styles.get(name, {}))
    ax.set_xlabel('k')
    ax.set_ylabel('Mean Recall@k')
    ax.set_xscale('log')
    ax.legend(loc='lower right')
    save_fig(fig, 'fig_4_4_recall_curve')